# 03 Gamma Forecast Impact

This notebook quantifies how the selected Gamma RPF case can inflate 7-day-ahead forecast error. Fig01 uses 2024-09-01 to 2024-09-07, the start of the September 2024 forecast test period. Smoke mode writes deterministic placeholder forecasts for layout checks without training LR or XGBoost; full forecast mode writes real model forecasts.


## 1. Imports And Paths

Load config and confirm the final Gamma dataset path.


In [ ]:
from pathlib import Path
import sys

# Keep notebook imports stable whether the notebook is run from JupyterLab,
# VS Code, or the repository root.
article_root = Path.cwd()
while article_root.name != "2_journal_article":
    if article_root.parent == article_root:
        raise RuntimeError("Could not locate publication/2_journal_article")
    article_root = article_root.parent
notebook_dir = article_root / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import _experiment_helpers as h

cfg = h.load_config(article_root)
paths = h.article_paths(article_root, cfg)
h.ensure_output_dirs(paths)
print(f"Article root: {article_root}")
print(f"Config schema: {cfg['schema_version']}")
print(f"Output root: {paths.outputs}")

print(article_root / cfg["paths"]["gamma_dataset_path"])
print(f"Full forecast: {cfg['execution']['run_full_forecast']}")


## 2. Load Gamma And Confirm Scope

Gamma must contain exactly one site over the one-year Beta window.


In [ ]:
gamma = h.load_dataset(article_root, cfg, "gamma")
site = gamma["substation_id"].iloc[0]
print(f"Gamma site: {site}")
h.dataset_summary(gamma, "Gamma")


## 3. Run Forecast-Impact Workflow

The workflow writes the Gamma series, perfect-model baseline, forecast rows, metrics, curated tables, figures, and manifest. Perfect-model baseline rows are deterministic comparisons against manually corrected data. Smoke forecast-model rows are marked as placeholders; paper-ready model results require full forecast mode.


In [ ]:
result = h.run_gamma_forecast_impact(article_root)
print(result["status"], result["gamma_site"])
result["metrics"]


## 4. Interpret The Perfect-Model Baseline

The `perfect_model_baseline` rows answer a narrow but powerful question: if a model perfectly predicted each data condition, how much RMSE remains against manually corrected data? The uncorrected-data row isolates RPF sign-error impact; the m8_xgb-corrected row reflects remaining correction error; the manually corrected row should be zero.


In [ ]:
result["metrics"].sort_values(["data_condition", "model"])
